In [ ]:

data = {
    'Review': [
        "At McDonald's the food was ok and the service was bad.",
        'I would not recommend this Japanese restaurant to anyone.',
        'I loved this restaurant when I traveled to Thailand last summer.',
        'The menu of Loving has a wide variety of options.',
        "The staff was friendly and helpful at Google's employees restaurant.",
        'The ambiance at Bella Italia is amazing, and the pasta dishes are delicious.',
        'I had a terrible experience at Pizza Hut. The pizza was burnt, and the service was slow.',
        'The sushi at Sushi Express is always fresh and flavorful.',
        'The steakhouse on Main Street has a cozy atmosphere and excellent steaks.',
        'The dessert selection at Sweet Treats is to die for!'
    ]
}

import pandas as pd
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

# Create DataFrame
df_raw = pd.DataFrame(data)
df_raw.head()

In [ ]:
# 1. Preprocessing function
def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Tokenize
    tokens = word_tokenize(text)
    # Remove punctuation
    tokens = [t for t in tokens if t not in string.punctuation]
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [t for t in tokens if t not in stop_words]
    # Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Apply preprocessing to dataset
df_raw['Preprocessed'] = df_raw['Review'].apply(preprocess_text)
df_raw[['Review', 'Preprocessed']].head()

In [ ]:
# 2. Create new dataset with cleaned text
df_clean = df_raw[['Preprocessed']].copy()
df_clean.head()

In [ ]:
# 3. NER function using spaCy
def perform_ner(text):
    import spacy
    nlp = spacy.load('en_core_web_sm')
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

# Test NER on raw and preprocessed data
print('Raw:', perform_ner(df_raw['Review'][0]))
print('Preprocessed:', perform_ner(df_raw['Preprocessed'][0]))

In [ ]:
# 4. POS tagging function using NLTK
def perform_pos_tagging(text):
    tokens = word_tokenize(text)
    return nltk.pos_tag(tokens)

# Test POS tagging on raw and preprocessed data
print('Raw:', perform_pos_tagging(df_raw['Review'][0]))
print('Preprocessed:', perform_pos_tagging(df_raw['Preprocessed'][0]))

In [ ]:
# 5. Apply NER and POS tagging to all data
# Apply to raw data
ner_raw = df_raw['Review'].apply(perform_ner)
pos_raw = df_raw['Review'].apply(perform_pos_tagging)
# Apply to preprocessed data
ner_clean = df_raw['Preprocessed'].apply(perform_ner)
pos_clean = df_raw['Preprocessed'].apply(perform_pos_tagging)

# Display results for first 3 reviews
for i in range(3):
    print(f"\nReview {i+1} (Raw): {df_raw['Review'][i]}")
    print("NER:", ner_raw[i])
    print("POS:", pos_raw[i])
    print(f"Preprocessed: {df_raw['Preprocessed'][i]}")
    print("NER:", ner_clean[i])
    print("POS:", pos_clean[i])

In [ ]:
# Exercise 2: Word Embeddings with Word2Vec
from gensim.models import Word2Vec

# Tokenize preprocessed text for Word2Vec
sentences = [row.split() for row in df_raw['Preprocessed']]

# 1. Train Word2Vec model
w2v_model = Word2Vec(sentences, vector_size=50, window=3, min_count=1, workers=2, seed=42)

# Print dimensions
print('Vocabulary size:', len(w2v_model.wv))
print('Vector dimension:', w2v_model.vector_size)
print('Example vector for "restaurant":', w2v_model.wv['restaurant'])

In [ ]:
# 2. Plotting word embeddings (first 2 dimensions)
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

def plot_word_embeddings(w2v_model):
    words = list(w2v_model.wv.index_to_key)
    X = w2v_model.wv[words]
    # Reduce to 2D for visualization
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)
    plt.figure(figsize=(10, 8))
    plt.scatter(X_pca[:, 0], X_pca[:, 1])
    for i, word in enumerate(words):
        plt.annotate(word, (X_pca[i, 0], X_pca[i, 1]))
    plt.title('Word Embeddings (PCA 2D)')
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.grid(True)
    plt.show()

# Call the function to plot
plot_word_embeddings(w2v_model)